# Distant Supervision for MNIST Digit Addition

This notebook demonstrates how to train a digit recognizer from weak supervision using `sltn`.

## Problem

Each training example is a pair of MNIST images, denoted by $x_1$ and $x_2$. The only observed label is their sum:

$$
s = d_1 + d_2,
$$

where $d_1$ and $d_2$ are the unknown digit labels of the two images.

The individual digit labels are not used during training. Instead, the model learns a neural function

$$
digit : Image \to Natnum
$$

by satisfying the arithmetic constraint:

$$
\forall x_1, x_2, s \;:\; digit(x_1) + digit(x_2) == s.
$$

This is a distant-supervision setting: supervision is provided only through the relation between two latent predictions and an observed aggregate label.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import torch
import numpy as np

torch.manual_seed(0)
np.random.seed(0)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## Dataset: MNIST pairs

We construct a paired MNIST dataset from the standard MNIST images.

For each pair:

- the first image is denoted by $x_1$;
- the second image is denoted by $x_2$;
- the training target is the sum $s$ of the two true digit labels.

The individual digit labels are used only to construct the sum labels. They are not provided to the training objective. The resulting dataset contains tensors of image pairs and their corresponding sum labels.

In [ ]:
import torchvision

def get_mnist_pairs(n_train=30000, n_test=5000):
    mnist_train = torchvision.datasets.MNIST(
        "../datasets/", train=True, download=True,
        transform=torchvision.transforms.ToTensor())
    mnist_test  = torchvision.datasets.MNIST(
        "../datasets/", train=False, download=True,
        transform=torchvision.transforms.ToTensor())

    def make_pair_set(imgs, lbls, n):
        imgs, lbls = imgs[:n * 2], lbls[:n * 2]
        imgs = imgs.float() / 255.0
        imgs = torch.unsqueeze(imgs, 1)
        pair_imgs = torch.stack([imgs[:n], imgs[n:2*n]], dim=1)
        pair_sum  = lbls[:n] + lbls[n:2*n]
        return pair_imgs, pair_sum.unsqueeze(-1)

    train_set = make_pair_set(mnist_train.data,  mnist_train.targets, n_train)
    test_set  = make_pair_set(mnist_test.data,  mnist_test.targets,  n_test)
    return train_set, test_set

train_set, test_set = get_mnist_pairs()

## 1. Signature

The signature declares the symbolic vocabulary used to express the digit-addition constraint.

This notebook uses two sorts:

- `Image`, representing a single MNIST image;
- `Natnum`, representing a scalar numerical value.

It declares:

- a function `digit : Image → Natnum`;
- an infix function `+ : Natnum × Natnum → Natnum`;
- an infix predicate `== : Natnum × Natnum → Bool`;
- variables `x1`, `x2`, and `s`, representing the two images and their observed sum.

In [ ]:
import sltn
from sltn import Signature, Interpretation

sig = Signature("digit_addition")

sig.sort("Image")
sig.sort('Natnum')  # auxiliary sort for the domain of d1 and d2

sig.function("digit", ["Image"], "Natnum")  # auxiliary function to interpret IsDigit(x, d) as
sig.function("+", ["Natnum", "Natnum"], "Natnum", infix=True)  # auxiliary function to interpret SumGuard(d1,

sig.predicate("==",["Natnum", "Natnum"], infix=True)  # auxiliary predicate to interpret Eq(d1 + d2, s) as

sig.variable("x1", "Image")
sig.variable("x2", "Image")
sig.variable("s", "Natnum")

print(sig)

## 2. Formula

The learning problem is specified by one universally quantified formula:

$$
\forall (x_1, x_2, s) \;:\; digit(x_1) + digit(x_2) == s.
$$

This formula states that, for every observed pair of images, the sum of the two predicted digit values should match the observed sum label. The digit recognizer is learned indirectly through the consistency of its predictions with the observed sums.

In [ ]:
phi = sig.parse("∀ (x1,x2,s) : digit(x1) + digit(x2) == s")
phi

## 3. Neural model: CNN digit function

The function `digit` is grounded by a convolutional neural network. The CNN maps each MNIST image to a probability distribution over the ten digit classes using a softmax output layer. To obtain a scalar digit value, the model computes the expected digit under that distribution:

$$
digit(x) = \sum_{k=0}^{9} k \, p_k(x),
$$

where $p_k(x)$ is the predicted probability of digit $k$ for image $x$. This expected-value representation is differentiable, which allows gradients from the sum constraint to update the CNN parameters.

In [ ]:
import torch
import torch.nn.functional as F

class DigitCNN(torch.nn.Module):
    """Simple CNN for digit classification."""
    def __init__(self):
        super().__init__()
        self.conv1 = torch.nn.Conv2d(1, 16, kernel_size=3, padding=1)
        self.conv2 = torch.nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.fc1   = torch.nn.Linear(32 * 7 * 7, 128)
        self.fc2   = torch.nn.Linear(128, 10)

    def forward(self, x):
        x = torch.relu(self.conv1(x))
        x = torch.max_pool2d(x, 2)
        x = torch.relu(self.conv2(x))
        x = torch.max_pool2d(x, 2)
        x = x.view(x.size(0), -1)
        x = torch.relu(self.fc1(x))
        x = torch.softmax(self.fc2(x), dim=-1)
        y = torch.sum(x * torch.arange(10, device=x.device), dim=-1, keepdim=True)
        return y

cnn = DigitCNN()

class DataLoader:
    """Simple shuffling data loader that yields (imgs, sums) tuples."""
    def __init__(self, fold, batch_size=32, shuffle=True):
        self.fold = fold
        self.batch_size = batch_size
        self.shuffle = shuffle

    def __len__(self):
        return int(np.ceil(self.fold[0].shape[0] / self.batch_size))

    def __iter__(self):
        n = self.fold[0].shape[0]
        idxlist = list(range(n))
        if self.shuffle:
            np.random.shuffle(idxlist)
        for start in range(0, n, self.batch_size):
            idx = idxlist[start:start + self.batch_size]
            yield self.fold[0][idx].to(device), self.fold[1][idx].to(device)

## 4. Interpretation

The interpretation grounds the symbolic vocabulary in concrete computational objects.

In this notebook:

- `Image` is interpreted as a tensor type with shape $(1, 28, 28)$;
- `Natnum` is interpreted as a scalar tensor type;
- `digit` is interpreted as the CNN defined above;
- `+` is interpreted as ordinary tensor addition;
- `==` is interpreted as a smooth equality predicate based on the distance between two scalar values.

### Equality kernel

The equality predicate is implemented as a distance-based fuzzy truth value:

$$
Eq(a,b) = \exp(-|a-b|).
$$

This function returns $1$ when $a=b$ and decreases smoothly as the distance between $a$ and $b$ grows. The following plot visualizes the equality truth value as $b$ varies while $a$ is fixed.

In [ ]:
# plot torch.exp(-10 * torch.abs(a - b)) over linspace 1,10
import matplotlib.pyplot as plt

# plot 1 d fix value of a
# set size
plt.figure(figsize=(6, 3))
a = 5
b = torch.linspace(1, 10, 100)
C = torch.exp(-1 * torch.abs(a - b))
plt.plot(b.numpy(), C.numpy())
plt.xlabel('b')
plt.ylabel('exp(-1 * abs(a - b))')
plt.title('exp(-1 * abs(a - b)) for a=5')
plt.show()

In [ ]:
interp = Interpretation(sig).to(device)
interp["Image"] = sltn.Type("Image", shape=(1, 28, 28), axis_names=["channel", "height", "width"])
interp["Natnum"] = sltn.Type("Natnum", shape=(1,))
interp["digit"] = cnn
interp["+"] = lambda a, b: a + b
interp["=="] = lambda a, b: torch.exp(-1 * torch.abs(a - b))
print(interp)

## 5. Evaluation criteria

We evaluate the model in two complementary ways.

### Knowledge-base satisfaction

The satisfaction score measures how well the learned interpretation satisfies the formula:

$$
digit(x_1) + digit(x_2) == s.
$$

For each batch, the two images and the observed sum are assigned to `x1`, `x2`, and `s`, and the formula is evaluated under the current interpretation.

### Sum accuracy

For interpretability, we also compute a discrete sum accuracy. The predicted digit values are rounded to the nearest integer, added together, and compared with the observed sum label.

In [ ]:
def evaluate_sat(loader):
    """KB satisfaction level over a loader."""
    mean_sat = 0
    for imgs, sums in loader:
        interp["x1"] = imgs[:,0]
        interp["x2"] = imgs[:,1]
        interp["s"] = sums.float()
        mean_sat += interp(phi).value
    return mean_sat / len(loader)

def evaluate_accuracy(loader):
    mean_acc = 0.0
    for imgs, sums in loader:
        interp["x1"] = imgs[:,0]
        interp["x2"] = imgs[:,1]
        interp["s"] = sums.float()
        pred_x = torch.round(cnn(interp["x1"]))
        pred_y = torch.round(cnn(interp["x2"]))
        mean_acc += ((pred_x + pred_y) == interp["s"]).float().mean().item()
    return mean_acc / len(loader)

## 6. Training loop

Training maximizes the satisfaction of the digit-addition formula.

For each batch:

1. the first and second images are assigned to `x1` and `x2`;
2. the observed sum is assigned to `s`;
3. the formula is evaluated under the current interpretation;
4. the loss is defined as: $\mathcal{L} = 1 - \mathrm{SatAgg}(\phi);$
5. gradients are backpropagated through the fuzzy equality predicate, the addition function, and the CNN grounding of `digit`;
6. the optimizer updates the CNN parameters.

Although the CNN is never trained with individual digit labels, the arithmetic constraint provides enough signal for it to learn digit-level structure.

In [ ]:
from tqdm.notebook import tqdm
train_loader = DataLoader(train_set, batch_size=32, shuffle=True)
test_loader  = DataLoader(test_set,  batch_size=32, shuffle=False)
optimizer = torch.optim.Adam(cnn.parameters(), lr=0.001)

for epoch in range(10):
    train_loss = train_sat = train_acc = 0.0

    for imgs, sums in tqdm(train_loader, desc=f"Epoch {epoch}"):

        interp["x1"] = imgs[:,0]
        interp["x2"] = imgs[:,1]
        interp["s"] = sums.float()

        optimizer.zero_grad()

        # d1 and d2 are resolved automatically from interp["d1"] / interp["d2"]
        loss = 1 - interp(phi).value
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
        train_sat  += 1.0 - loss.item()

        with torch.no_grad():
            px1 = torch.round(cnn(interp["x1"]))
            px2 = torch.round(cnn(interp["x2"]))
            train_acc += ((px1 + px2) == interp["s"]).float().mean().item()

    n = len(train_loader)
    with torch.no_grad():
        test_sat = evaluate_sat(test_loader)
        test_acc = evaluate_accuracy(test_loader)

    print(
        f"epoch {epoch:2d} | loss {train_loss/n:.4f} "
        f"| Train Sat {train_sat/n:.3f} | Test Sat {test_sat.item():.3f} "
        f"| Train Acc {train_acc/n:.3f} | Test Acc {test_acc:.3f}"
    )

## 7. Sum prediction analysis

The first confusion matrix evaluates the end-to-end digit-addition task. For each test pair, the model predicts two digit values, rounds them to discrete digits, and computes their sum. The resulting predicted sum is compared against the true observed sum. This matrix shows which sums are predicted reliably and which sums are commonly confused.

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns

# Confusion matrix for the sum problem
# Predict sums using the model
with torch.no_grad():
    # test_set[0]: shape [N, 2, 1, 28, 28], test_set[1]: shape [N]
    imgs_sum = test_set[0].to(device)
    sums_true = test_set[1].to(device)
    pred_x1 = torch.round(cnn(imgs_sum[:,0])).squeeze(-1)
    pred_x2 = torch.round(cnn(imgs_sum[:,1])).squeeze(-1)
    sums_pred = (pred_x1 + pred_x2).long()

cm_sum = confusion_matrix(sums_true.cpu().numpy(), sums_pred.cpu().numpy())
plt.figure(figsize=(16, 8))
sns.heatmap(cm_sum, annot=True, cmap='Blues')
plt.xlabel('Predicted Sum')
plt.ylabel('True Sum')
plt.title('Confusion Matrix: Sum Prediction')
plt.show()

## 8. Single-digit classification

Although individual digit labels are not used during training, the learned `digit` function can be evaluated on standard single MNIST images.

This section measures how well the CNN has recovered a digit classifier from sum-level supervision alone. The reported accuracy and confusion matrix are not part of the training objective; they are diagnostic tools for understanding what the learned latent digit function represents.

In [ ]:
# test the accuracy of digit alone
# Load MNIST test set for single digit classification
mnist_test = torchvision.datasets.MNIST(
    "../datasets/", train=False, download=True,
    transform=torchvision.transforms.ToTensor()
)

test_imgs = mnist_test.data.unsqueeze(1).float().to(device) / 255.0
test_labels = mnist_test.targets.to(device)

# Predict digits
with torch.no_grad():
    preds = torch.round(cnn(test_imgs)).squeeze(-1).long()
    acc = (preds == test_labels).float().mean().item()

print(f"Single digit test accuracy: {acc:.4f}")

# Confusion matrix for single digit classification
cm_digit = confusion_matrix(test_labels.cpu().numpy(), preds.cpu().numpy())
plt.figure(figsize=(8, 6))
sns.heatmap(cm_digit, annot=True, fmt='d', cmap='Blues', cbar=False)
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix: Single Digit Classification')
plt.show()